In [128]:
import pandas as pd, numpy as np
from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = Path("data")       
HOURS_PER_TIME_UNIT_DAY = 24

# Get the Data

In [129]:
portfolio_raw  = pd.read_json(DATA_DIR / "portfolio.json",  lines=True)
profile_raw    = pd.read_json(DATA_DIR / "profile.json",    lines=True)
transcript_raw = pd.read_json(DATA_DIR / "transcript.json", lines=True)

for name, df in [("portfolio", portfolio_raw), ("profile", profile_raw), ("transcript", transcript_raw)]:
    print(f"{name:<11} - [{df.shape}] || rows={len(df):>7,} || cols={list(df.columns)}")

portfolio   - [(10, 6)] || rows=     10 || cols=['reward', 'channels', 'difficulty', 'duration', 'offer_type', 'id']
profile     - [(17000, 5)] || rows= 17,000 || cols=['gender', 'age', 'id', 'became_member_on', 'income']
transcript  - [(306534, 4)] || rows=306,534 || cols=['person', 'event', 'value', 'time']


# Data Exploration & Cleaning

## Portfolio Data

In [130]:
portfolio_raw.head()

,reward,channels,difficulty,duration,offer_type,id
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"[web, email, mobile]",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"[web, email]",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [131]:
# Check for duplicates in column 'id'

portfolio_raw['id'].duplicated().any()

np.False_

In [132]:
# Create a copy of the df

portfolio = portfolio_raw.rename(columns={"id": "offer_id", "reward": "offer_reward"}).copy()

portfolio.head(3)

,offer_reward,channels,difficulty,duration,offer_type,offer_id
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed


In [133]:
for ch in ["email", "mobile", "social", "web"]:
    portfolio[f"ch_{ch}"] = portfolio["channels"].apply(lambda L: int(ch in L))

portfolio.head(3)

,offer_reward,channels,difficulty,duration,offer_type,offer_id,ch_email,ch_mobile,ch_social,ch_web
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,1,1,1,0
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,0,1


In [134]:
portfolio["offer_label"] = (portfolio["offer_type"].str.upper().str[:4] + "_r" + portfolio["offer_reward"].astype(str)
                            + "_d" + portfolio["difficulty"].astype(str) + "_" + portfolio["duration"].astype(int).astype(str) + "d")

portfolio.head(3)                          

,offer_reward,channels,difficulty,duration,offer_type,offer_id,ch_email,ch_mobile,ch_social,ch_web,offer_label
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,1,1,1,0,BOGO_r10_d10_7d
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1,BOGO_r10_d10_5d
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,0,1,INFO_r0_d0_4d


In [135]:
assert portfolio["offer_id"].is_unique and portfolio["offer_label"].is_unique

portfolio[["offer_label", "offer_type", "offer_reward", "difficulty", "duration", "channels", "offer_id"]].sort_values("offer_label")

,offer_label,offer_type,offer_reward,difficulty,duration,channels,offer_id
1,BOGO_r10_d10_5d,bogo,10,10,5,"[web, email, mobile, social]",4d5c57ea9a6940dd891ad53e9dbe8da0
0,BOGO_r10_d10_7d,bogo,10,10,7,"[email, mobile, social]",ae264e3637204a6fb9bb56bc8210ddfd
8,BOGO_r5_d5_5d,bogo,5,5,5,"[web, email, mobile, social]",f19421c1d4aa40978ebb69ca19b0e20d
3,BOGO_r5_d5_7d,bogo,5,5,7,"[web, email, mobile]",9b98b8c7a33c4b65b9aebfe6a799e6d9
6,DISC_r2_d10_10d,discount,2,10,10,"[web, email, mobile, social]",fafdcd668e3743c1bb461111dcafc2a4
9,DISC_r2_d10_7d,discount,2,10,7,"[web, email, mobile]",2906b810c7d4411798c6938adc9daaa5
5,DISC_r3_d7_7d,discount,3,7,7,"[web, email, mobile, social]",2298d6c36e964ae4a3e7e9706d1fb8c2
4,DISC_r5_d20_10d,discount,5,20,10,"[web, email]",0b1e1539f2cc45b7b9fa7c272da2e1d7
7,INFO_r0_d0_3d,informational,0,0,3,"[email, mobile, social]",5a8bc65990b245e5a138643cd4eb9837
2,INFO_r0_d0_4d,informational,0,0,4,"[web, email, mobile]",3f207df678b143eea3cee63160fa8bed


## Profile Data

In [136]:
profile_raw.head(5)

,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,"112,000.00"
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,"100,000.00"
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


In [137]:
profile_raw['id'].duplicated().any()

np.False_

In [138]:
# Create a copy of the data

profile = profile_raw.rename(columns={"id": "person"}).copy()

In [139]:
# Change the dtaa format for the date field

profile["became_member_on"] = pd.to_datetime(profile["became_member_on"].astype(str), format="%Y%m%d", errors="coerce")

In [140]:
profile["profile_incomplete"] = profile["age"].eq(118) & profile["gender"].isna() & profile["income"].isna()

profile["age"] = profile["age"].where(profile["age"].ne(118))      # 118 is a placeholder, not an age

profile.head()

,gender,age,person,became_member_on,income,profile_incomplete
0,NaN,NaN,68be06ca386d4c31939f3a4f0e3dd783,2017-02-12,NaN,True
1,F,55.00,0610b486422d4921ae7d2bf64640c50b,2017-07-15,"112,000.00",False
2,NaN,NaN,38fe809add3b4fcf9315a9694bb96ff5,2018-07-12,NaN,True
3,F,75.00,78afa995795e4d85b5d9ceeca43f5fef,2017-05-09,"100,000.00",False
4,NaN,NaN,a03223e636434f42ac4c3df47e8bac43,2017-08-04,NaN,True


## Transcript Data

In [141]:
transcript_raw.head()

,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


In [142]:
transcripts = transcript_raw.copy()

In [143]:
# The values in column "value" seem to be JSON objects or dictionaries, we have to flatten it

unique_keys = set(
    key
    for item in transcripts['value']
    if isinstance(item, dict)
    for key in item.keys()
)

print(unique_keys)

transcript_values = pd.json_normalize(transcripts["value"].tolist())
transcript_values.head()

{'offer id', 'reward', 'offer_id', 'amount'}


,offer id,amount,offer_id,reward
0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,NaN
1,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN
2,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,NaN
3,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN
4,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,NaN


In [144]:
for col in ["offer id", "offer_id", "amount", "reward"]:
    if col not in v: v[col] = np.nan

both_keys = (v["offer id"].notna() & v["offer_id"].notna()).sum()

both_keys

np.int64(0)

In [145]:
transcripts["offer_id"]    = transcript_values["offer id"].combine_first(v["offer_id"]).values   # received/viewed use 'offer id', completed uses 'offer_id'
transcripts["amount"]      = transcript_values["amount"].values
transcripts["reward_paid"] = transcript_values["reward"].values
transcripts["event"]       = transcripts["event"].str.replace(" ", "_")

# transcripts = transcripts.drop(columns="value")

transcripts["event_id"] = np.arange(len(t))

print("rows with BOTH 'offer id' and 'offer_id':", both_keys)

transcripts["event"].value_counts()

transcripts.head()

rows with BOTH 'offer id' and 'offer_id': 0


,person,event,value,time,offer_id,amount,reward_paid,event_id
0,78afa995795e4d85b5d9ceeca43f5fef,offer_received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,0
1,a03223e636434f42ac4c3df47e8bac43,offer_received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,1
2,e2127556f4f64592b11af22de27a7932,offer_received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,2
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer_received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,3
4,68617ca6246f4fbc85e91a2a49552598,offer_received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,4


# Data Quality Checks

In [146]:
def dq(check, observed, expected, ok):
    return {"check": check, "observed": observed, "expected": expected, "status": "PASS" if ok else "WARN"}

In [147]:
tx   = transcripts[transcripts.event == "transaction"]

offr = transcripts[transcripts.event != "transaction"]

dup_mask = transcripts.duplicated(subset=["person", "event", "offer_id", "amount", "time"], keep="first")

comp_ev = transcripts[transcripts.event == "offer_completed"].merge(portfolio[["offer_id", "offer_reward", "offer_type"]], on="offer_id", how="left")

n_bad_reward = int((comp_ev.reward_paid != comp_ev.offer_reward).sum())

n_info_comp  = int(comp_ev.offer_type.eq("informational").sum())

n_comp_no_tx = int((~comp_ev.set_index(["person", "time"]).index.isin(tx.set_index(["person", "time"]).index)).sum())

recv_times = sorted(int(x) for x in t.loc[t.event == "offer_received", "time"].unique())

In [148]:
checks = [
    dq("Row counts (portfolio/profile/transcript)", f"{len(portfolio)}/{len(profile)}/{len(t)}", "10/17000/306534",
       (len(portfolio), len(profile), len(t)) == (10, 17000, 306534)),
    dq("Unique keys: offer_id, person", f"{portfolio.offer_id.is_unique}, {profile.person.is_unique}", "True, True",
       portfolio.offer_id.is_unique and profile.person.is_unique),
    dq("Transcript members missing from profile", (~t.person.isin(profile.person)).sum(), 0, (~t.person.isin(profile.person)).sum() == 0),
    dq("Offer IDs missing from portfolio", (~offr.offer_id.isin(portfolio.offer_id)).sum(), 0, (~offr.offer_id.isin(portfolio.offer_id)).sum() == 0),
    dq("Transactions with no amount OR with an offer id", (tx.amount.isna() | tx.offer_id.notna()).sum(), 0, (tx.amount.isna() | tx.offer_id.notna()).sum() == 0),
    dq("Offer events with no offer id OR with an amount", (offr.offer_id.isna() | offr.amount.notna()).sum(), 0, (offr.offer_id.isna() | offr.amount.notna()).sum() == 0),
    dq("Incomplete profiles (age 118 + no gender + no income)", int(profile.profile_incomplete.sum()), 2175, profile.profile_incomplete.sum() == 2175),
    dq("age==118 / gender null / income null all coincide",
       int((profile.age.isna() != profile.income.isna()).sum() + (profile.gender.isna() != profile.income.isna()).sum()), 0,
       (profile.age.isna() == profile.income.isna()).all() and (profile.gender.isna() == profile.income.isna()).all()),
    dq("Unparseable membership dates", int(profile.became_member_on.isna().sum()), 0, profile.became_member_on.isna().sum() == 0),
    dq("Exact duplicate event rows", int(dup_mask.sum()), "0 (or explained)", dup_mask.sum() == 0),
    dq("Non-positive transaction amounts", int((tx.amount <= 0).sum()), 0, (tx.amount <= 0).sum() == 0),
    dq("Completed reward != portfolio reward", n_bad_reward, 0, n_bad_reward == 0),
    dq("Completions of informational offers", n_info_comp, 0, n_info_comp == 0),
    dq("Completion events with no same-time transaction", n_comp_no_tx, 0, n_comp_no_tx == 0),
    dq("Time range (min-max)", f"{t.time.min()}-{t.time.max()}", "assumed hours, ~30 days", True),
    dq("Offer send times (gaps multiples of 24 if hours)", recv_times, "multiples of 24", all(x % 24 == 0 for x in recv_times)),
    dq("Members with no events at all", int((~profile.person.isin(t.person)).sum()), "small", True),
]
dq_table = pd.DataFrame(checks)
print("Duplicates by event type:\n", t[dup_mask].event.value_counts())
dq_table

Duplicates by event type:
 event
offer_completed    397
Name: count, dtype: int64


,check,observed,expected,status
0,Row counts (portfolio/profile/transcript),10/17000/306534,10/17000/306534,PASS
1,"Unique keys: offer_id, person","True, True","True, True",PASS
2,Transcript members missing from profile,0,0,PASS
3,Offer IDs missing from portfolio,0,0,PASS
4,Transactions with no amount OR with an offer id,0,0,PASS
5,Offer events with no offer id OR with an amount,0,0,PASS
6,Incomplete profiles (age 118 + no gender + no ...,2175,2175,PASS
7,age==118 / gender null / income null all coincide,0,0,PASS
8,Unparseable membership dates,0,0,PASS
9,Exact duplicate event rows,397,0 (or explained),WARN


In [149]:
# ---- CELL 6: offer-instance table (grain: 1 row per offer RECEIVED)
rec = (t[t.event == "offer_received"][["person", "offer_id", "time"]]
       .rename(columns={"time": "t_received"})
       .merge(portfolio[["offer_id", "offer_label", "offer_type", "offer_reward", "difficulty", "duration"]],
              on="offer_id", how="left", validate="many_to_one")
       .reset_index(drop=True))
rec["instance_id"] = np.arange(len(rec))
rec["t_expiry"] = rec["t_received"] + rec["duration"] * HOURS_PER_TIME_UNIT_DAY

def match_to_instances(event_name):
    """Match an offer event to every received instance of the same member+offer whose validity window contains it."""
    ev = t[t.event == event_name][["event_id", "person", "offer_id", "time"]]
    m = rec[["instance_id", "person", "offer_id", "t_received", "t_expiry"]].merge(ev, on=["person", "offer_id"])
    m = m[(m.time >= m.t_received) & (m.time <= m.t_expiry)]
    orphans = ev.loc[~ev.event_id.isin(m.event_id)]
    return m, orphans

views, orphan_views = match_to_instances("offer_viewed")
comps, orphan_comps = match_to_instances("offer_completed")
rec["t_viewed"]    = rec.instance_id.map(views.groupby("instance_id").time.min())
rec["t_completed"] = rec.instance_id.map(comps.groupby("instance_id").time.min())

is_info = rec.offer_type.eq("informational")
rec["viewed"]               = rec.t_viewed.notna()
rec["completed"]            = rec.t_completed.notna() & ~is_info
rec["completed_after_view"] = rec.completed & rec.viewed & (rec.t_viewed <= rec.t_completed)
rec["completed_unaware"]    = rec.completed & ~rec.completed_after_view     # never viewed, or viewed only after completing
rec["reward_cost"]          = np.where(rec.completed, rec.offer_reward, 0)

# same member+offer received again while the previous one was still valid
rec = rec.sort_values(["person", "offer_id", "t_received"])
rec["overlaps_previous"] = rec.t_received <= rec.groupby(["person", "offer_id"]).t_expiry.shift()
rec = rec.sort_values("instance_id")

# spend while the instance was valid (CONTEXT ONLY - other offers may be active simultaneously)
w = rec[["instance_id", "person", "t_received", "t_expiry"]].merge(tx[["person", "time", "amount"]], on="person")
w = w[(w.time >= w.t_received) & (w.time <= w.t_expiry)]
rec["spend_in_window"] = rec.instance_id.map(w.groupby("instance_id").amount.sum()).fillna(0)
a = rec[["instance_id", "person", "t_received"]].merge(
        rec[["person", "t_received", "t_expiry"]].rename(columns={"t_received": "o_start", "t_expiry": "o_end"}), on="person")
rec["n_offers_active_at_receipt"] = rec.instance_id.map(          # includes the instance itself
    a[(a.o_start <= a.t_received) & (a.t_received <= a.o_end)].groupby("instance_id").size())

print(pd.Series({
    "received instances": len(rec),
    "views not matched to a valid instance": len(orphan_views),
    "completions not matched to a valid instance": len(orphan_comps),
    "completion EVENTS matched to >1 instance": int((comps.groupby('event_id').size() > 1).sum()),
    "instances overlapping a previous identical offer": int(rec.overlaps_previous.sum()),
    "completed instances, never viewed in window": int((rec.completed & ~rec.viewed).sum()),
    "completed instances, viewed only AFTER completing": int((rec.completed & rec.viewed & ~rec.completed_after_view).sum()),
}))

received instances                                   76277
views not matched to a valid instance                 1158
completions not matched to a valid instance              0
completion EVENTS matched to >1 instance              1776
instances overlapping a previous identical offer      5295
completed instances, never viewed in window           5734
completed instances, viewed only AFTER completing     4220
dtype: int64


In [150]:
# ---- CELL 7: member table (grain: 1 row per member) + privacy-safe BEHAVIOURAL cohorts
member = profile[["person", "profile_incomplete"]].copy()
member = member.merge(tx.groupby("person").agg(n_tx=("amount", "size"), spend=("amount", "sum")).reset_index(),
                      on="person", how="left", validate="one_to_one")
member = member.merge(rec.groupby("person").agg(n_received=("instance_id", "size"), n_viewed=("viewed", "sum"),
                                                n_completed=("completed", "sum"), n_unaware=("completed_unaware", "sum"),
                                                reward_cost=("reward_cost", "sum")).reset_index(),
                      on="person", how="left", validate="one_to_one")
member = member.fillna({c: 0 for c in ["n_tx", "spend", "n_received", "n_viewed", "n_completed", "n_unaware", "reward_cost"]})

member["frequency_cohort"] = pd.cut(member.n_tx, bins=[-1, 0, 3, 8, np.inf],
                                    labels=["F0 no purchase", "F1 1-3 purchases", "F2 4-8 purchases", "F3 9+ purchases"])
view_share = member.n_viewed / member.n_received.replace(0, np.nan)
member["engagement_cohort"] = np.select([member.n_received.eq(0), view_share >= 0.75, view_share >= 0.25],
                                        ["E0 no offers", "E3 views most offers", "E2 views some offers"],
                                        default="E1 rarely views offers")
rec = rec.merge(member[["person", "frequency_cohort", "engagement_cohort"]], on="person", how="left", validate="many_to_one")
print(member.frequency_cohort.value_counts().sort_index(), "\n")
print(member.engagement_cohort.value_counts().sort_index())

frequency_cohort
F0 no purchase       422
F1 1-3 purchases    2501
F2 4-8 purchases    7265
F3 9+ purchases     6812
Name: count, dtype: int64 

engagement_cohort
E0 no offers                  6
E1 rarely views offers      428
E2 views some offers       6311
E3 views most offers      10255
Name: count, dtype: int64


In [151]:
# ---- CELL 8: metric sheet with explicit numerators and denominators
def metric(name, num, den, definition):
    return {"metric": name, "numerator": num, "denominator": den, "value": num / den if den else np.nan, "definition": definition}

n_members, n_purchasers = len(member), int((member.n_tx > 0).sum())
total_spend, n_tx = tx.amount.sum(), len(tx)
promo = rec[~rec.offer_type.eq("informational")]
recorded_reward = t.loc[t.event == "offer_completed", "reward_paid"].sum()

metric_sheet = pd.DataFrame([
    metric("Purchasing-member rate", n_purchasers, n_members, "members with >=1 transaction / all profile members"),
    metric("Spend per member", total_spend, n_members, "total transaction amount / all profile members"),
    metric("Spend per purchasing member", total_spend, n_purchasers, "total transaction amount / members with >=1 transaction"),
    metric("Average transaction value", total_spend, n_tx, "total transaction amount / transactions"),
    metric("Offer view rate (all offers)", rec.viewed.sum(), len(rec), "instances viewed within validity / instances received"),
    metric("Completion rate (BOGO+discount)", promo.completed.sum(), len(promo), "promo instances completed within validity / promo instances received"),
    metric("Unaware-completion share", promo.completed_unaware.sum(), promo.completed.sum(), "completed with no view before completion / completed instances"),
    metric("Recorded reward face value as % of spend", recorded_reward, total_spend, "sum of reward on completed EVENTS / total transaction amount"),
])
metric_sheet

,metric,numerator,denominator,value,definition
0,Purchasing-member rate,"16,578.00","17,000.00",0.98,members with >=1 transaction / all profile mem...
1,Spend per member,"1,775,451.97","17,000.00",104.44,total transaction amount / all profile members
2,Spend per purchasing member,"1,775,451.97","16,578.00",107.10,total transaction amount / members with >=1 tr...
3,Average transaction value,"1,775,451.97","138,953.00",12.78,total transaction amount / transactions
4,Offer view rate (all offers),"56,895.00","76,277.00",0.75,instances viewed within validity / instances r...
5,Completion rate (BOGO+discount),"33,631.00","61,042.00",0.55,promo instances completed within validity / pr...
6,Unaware-completion share,"9,954.00","33,631.00",0.30,completed with no view before completion / com...
7,Recorded reward face value as % of spend,"164,676.00","1,775,451.97",0.09,sum of reward on completed EVENTS / total tran...


In [152]:
# ---- CELL 9: comparisons - offer type, offer, behavioural cohorts, channel
def offer_kpis(df, by):
    g = df.groupby(by, observed=True)
    out = pd.DataFrame({
        "received":            g.size(),
        "members":             g.person.nunique(),
        "view_rate":           g.viewed.mean(),
        "completion_rate":     g.completed.sum() / g.size(),
        "unaware_share":       g.completed_unaware.sum() / g.completed.sum().replace(0, np.nan),
        "reward_cost":         g.reward_cost.sum(),
        "reward_per_received": g.reward_cost.sum() / g.size(),
        "unaware_reward_cost": g.apply(lambda x: x.loc[x.completed_unaware, "reward_cost"].sum(), include_groups=False),
        "spend_in_window_per_received": g.spend_in_window.mean(),
    })
    out.loc[out.index.get_level_values(-1).astype(str).str.contains("informational|INFO"), ["completion_rate", "unaware_share"]] = np.nan
    return out

by_type   = offer_kpis(rec, "offer_type")
by_offer  = offer_kpis(rec, ["offer_type", "offer_label"])
by_engage = offer_kpis(rec[rec.offer_type != "informational"], "engagement_cohort")
by_freq   = offer_kpis(rec[rec.offer_type != "informational"], "frequency_cohort")
by_social = (rec.merge(portfolio[["offer_id", "ch_social"]], on="offer_id")
                .groupby(["offer_type", "ch_social"]).viewed.mean().unstack())   # association only

member_by_freq = member.groupby("frequency_cohort", observed=True).agg(
    members=("person", "size"), spend=("spend", "sum"), transactions=("n_tx", "sum"), reward_cost=("reward_cost", "sum"))
member_by_freq["share_of_members"]    = member_by_freq.members / n_members
member_by_freq["share_of_spend"]      = member_by_freq.spend / total_spend
member_by_freq["spend_per_member"]    = member_by_freq.spend / member_by_freq.members
member_by_freq["reward_pct_of_spend"] = member_by_freq.reward_cost / member_by_freq.spend.replace(0, np.nan)

member_by_profile = member.groupby("profile_incomplete").agg(members=("person", "size"),
    purchasing_rate=("n_tx", lambda s: (s > 0).mean()), spend_per_member=("spend", "mean"))
for x in [by_type, by_offer, by_engage, by_freq, by_social, member_by_freq, member_by_profile]: display(x)

,received,members,view_rate,completion_rate,unaware_share,reward_cost,reward_per_received,unaware_reward_cost,spend_in_window_per_received
offer_type,,,,,,,,,
bogo,30499,14992,0.83,0.51,0.29,113440,3.72,31045,26.51
discount,30543,14945,0.71,0.59,0.30,51409,1.68,17725,34.97
informational,15235,10547,0.65,NaN,NaN,0,0.00,0,13.80


received  members  view_rate  completion_rate  \
offer_type    offer_label                                                      
bogo          BOGO_r10_d10_5d      7593     6330       0.95             0.44   
              BOGO_r10_d10_7d      7658     6374       0.88             0.48   
              BOGO_r5_d5_5d        7571     6262       0.95             0.57   
              BOGO_r5_d5_7d        7677     6355       0.54             0.57   
discount      DISC_r2_d10_10d      7597     6332       0.97             0.70   
              DISC_r2_d10_7d       7632     6285       0.54             0.53   
              DISC_r3_d7_7d        7646     6325       0.96             0.68   
              DISC_r5_d20_10d      7668     6374       0.36             0.45   
informational INFO_r0_d0_3d        7618     6320       0.81              NaN   
              INFO_r0_d0_4d        7617     6331       0.50              NaN   

                               unaware_share  reward_cost  \
offer_type    offer_label                                   
bogo          BOGO_r10_d10_5d           0.17        33310   
              BOGO_r10_d10_7d           0.29        36880   
              BOGO_r5_d5_5d             0.17        21480   
              BOGO_r5_d5_7d             0.51        21770   
discount      DISC_r2_d10_10d           0.12        10664   
              DISC_r2_d10_7d            0.47         8050   
              DISC_r3_d7_7d             0.15        15495   
              DISC_r5_d20_10d           0.60        17200   
informational INFO_r0_d0_3d              NaN            0   
              INFO_r0_d0_4d              NaN            0   

                               reward_per_received  unaware_reward_cost  \
offer_type    offer_label                                                 
bogo          BOGO_r10_d10_5d                 4.39                 5620   
              BOGO_r10_d10_7d                 4.82                10580   
              BOGO_r5_d5_5d                   2.84                 3750   
              BOGO_r5_d5_7d                   2.84                11095   
discount      DISC_r2_d10_10d                 1.40                 1308   
              DISC_r2_d10_7d                  1.05                 3752   
              DISC_r3_d7_7d                   2.03                 2265   
              DISC_r5_d20_10d                 2.24                10400   
informational INFO_r0_d0_3d                   0.00                    0   
              INFO_r0_d0_4d                   0.00                    0   

                               spend_in_window_per_received  
offer_type    offer_label                                    
bogo          BOGO_r10_d10_5d                         24.72  
              BOGO_r10_d10_7d                         31.30  
              BOGO_r5_d5_5d                           24.34  
              BOGO_r5_d5_7d                           25.65  
discount      DISC_r2_d10_10d                         47.40  
              DISC_r2_d10_7d                          26.57  
              DISC_r3_d7_7d                           32.96  
              DISC_r5_d20_10d                         33.04  
informational INFO_r0_d0_3d                           13.28  
              INFO_r0_d0_4d                           14.33

,received,members,view_rate,completion_rate,unaware_share,reward_cost,reward_per_received,unaware_reward_cost,spend_in_window_per_received
engagement_cohort,,,,,,,,,
E1 rarely views offers,1375,419,0.14,0.31,0.81,2036,1.48,1650,14.63
E2 views some offers,22576,6292,0.56,0.48,0.46,51945,2.30,23395,26.63
E3 views most offers,37091,10217,0.92,0.60,0.21,110868,2.99,23725,33.85


,received,members,view_rate,completion_rate,unaware_share,reward_cost,reward_per_received,unaware_reward_cost,spend_in_window_per_received
frequency_cohort,,,,,,,,,
F0 no purchase,1532,418,0.80,0.00,NaN,0,0.00,0,0.00
F1 1-3 purchases,8543,2485,0.74,0.27,0.35,11482,1.34,4185,9.16
F2 4-8 purchases,24996,7220,0.74,0.51,0.31,64076,2.56,20026,24.20
F3 9+ purchases,25971,6805,0.80,0.72,0.28,89291,3.44,24559,45.97


ch_social,0,1
offer_type,,
bogo,0.54,0.93
discount,0.45,0.96
informational,0.50,0.81


,members,spend,transactions,reward_cost,share_of_members,share_of_spend,spend_per_member,reward_pct_of_spend
frequency_cohort,,,,,,,,
F0 no purchase,422,0.00,0.00,0.00,0.02,0.00,0.00,NaN
F1 1-3 purchases,2501,"85,494.46","5,861.00","11,482.00",0.15,0.05,34.18,0.13
F2 4-8 purchases,7265,"622,796.05","42,887.00","64,076.00",0.43,0.35,85.73,0.10
F3 9+ purchases,6812,"1,067,161.46","90,205.00","89,291.00",0.40,0.60,156.66,0.08


,members,purchasing_rate,spend_per_member
profile_incomplete,,,
False,14825,0.98,117.03
True,2175,0.96,18.63


In [153]:
# ---- CELL 10: reconciliation (proves nothing lost or double counted)
pd.Series({
    "transaction events": n_tx, "transactions in member table": int(member.n_tx.sum()),
    "spend: transcript": round(total_spend, 2), "spend: member table": round(member.spend.sum(), 2),
    "completion events": int((t.event == "offer_completed").sum()), "completed instances": int(rec.completed.sum()),
    "reward: completion events": recorded_reward, "reward: completed instances": rec.reward_cost.sum(),
    "reward: excl. exact duplicate events": t.loc[(t.event == "offer_completed") & ~dup_mask, "reward_paid"].sum(),
})

transaction events                       138,953.00
transactions in member table             138,953.00
spend: transcript                      1,775,451.97
spend: member table                    1,775,451.97
completion events                         33,579.00
completed instances                       33,631.00
reward: completion events                164,676.00
reward: completed instances              164,849.00
reward: excl. exact duplicate events     162,679.00
dtype: float64